In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

# 3 digit terakhir NIM 2301010515 -> SEED = 515
SEED = 515
np.random.seed(SEED)

os.makedirs("../reports", exist_ok=True)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

print(f"SEED aktif: {SEED}")

--MASALAH DATA--

### 1. Pemahaman masalah dan data
- **Tujuan analisis:** Membangun model klasifikasi multi-kelas untuk memprediksi kategori status/kelas tanah (`Output`: `0`, `1`, `2`) berdasarkan 12 parameter kimia dan kesuburan tanah (`N`, `P`, `K`, `pH`, `EC`, `OC`, `S`, `Zn`, `Fe`, `Cu`, `Mn`, `B`).
- **Siapa yang memakai hasilnya:** Petani, penyuluh pertanian, agronom, atau pengelola lahan pertanian yang ingin mengklasifikasikan kondisi tanah secara cepat tanpa menunggu analisis lanjutan yang mahal.
- **Keputusan yang terbantu:** Membantu menentukan prioritas pemupukan, koreksi keasaman/salinitas tanah (`pH`, `EC`), atau penyesuaian komoditas/perlakuan lahan sesuai kelas tanah yang diprediksi.
- **Sumber data:** `dataset1.csv` — *isi link sumber dataset asli (mis. Kaggle/UCI/repositori sumber) dan tanggal akses aktual saat mengunduh data*.
- **Lisensi & privasi:** *Cantumkan lisensi dataset sesuai sumber aslinya*; data berupa parameter kimia tanah (bukan data identitas pribadi).

In [ ]:
df = pd.read_csv("../data/dataset1.csv")  # ubah ke "dataset1.csv" jika notebook berada di folder yang sama

print("Ukuran dataset (baris, kolom):", df.shape)
print("\nTipe data:\n", df.dtypes)

deskripsi_kolom = pd.DataFrame([
    {"Nama Kolom": "N", "Tipe": str(df["N"].dtype), "Arti": "Kandungan Nitrogen (N) tanah", "Satuan": "mg/kg (atau satuan dataset asli)"},
    {"Nama Kolom": "P", "Tipe": str(df["P"].dtype), "Arti": "Kandungan Fosfor (P) tanah", "Satuan": "mg/kg"},
    {"Nama Kolom": "K", "Tipe": str(df["K"].dtype), "Arti": "Kandungan Kalium (K) tanah", "Satuan": "mg/kg"},
    {"Nama Kolom": "pH", "Tipe": str(df["pH"].dtype), "Arti": "Tingkat keasaman/kebasaan tanah", "Satuan": "skala pH"},
    {"Nama Kolom": "EC", "Tipe": str(df["EC"].dtype), "Arti": "Electrical Conductivity / daya hantar listrik tanah", "Satuan": "dS/m"},
    {"Nama Kolom": "OC", "Tipe": str(df["OC"].dtype), "Arti": "Organic Carbon / karbon organik tanah", "Satuan": "%"},
    {"Nama Kolom": "S", "Tipe": str(df["S"].dtype), "Arti": "Kandungan Belerang / Sulfur (S) tanah", "Satuan": "mg/kg"},
    {"Nama Kolom": "Zn", "Tipe": str(df["Zn"].dtype), "Arti": "Kandungan Seng / Zinc (Zn) mikro", "Satuan": "mg/kg"},
    {"Nama Kolom": "Fe", "Tipe": str(df["Fe"].dtype), "Arti": "Kandungan Besi / Iron (Fe) mikro", "Satuan": "mg/kg"},
    {"Nama Kolom": "Cu", "Tipe": str(df["Cu"].dtype), "Arti": "Kandungan Tembaga / Copper (Cu) mikro", "Satuan": "mg/kg"},
    {"Nama Kolom": "Mn", "Tipe": str(df["Mn"].dtype), "Arti": "Kandungan Mangan (Mn) mikro", "Satuan": "mg/kg"},
    {"Nama Kolom": "B", "Tipe": str(df["B"].dtype), "Arti": "Kandungan Boron (B) mikro", "Satuan": "mg/kg"},
    {"Nama Kolom": "Output", "Tipe": str(df["Output"].dtype), "Arti": "Target kelas tanah (0, 1, 2)", "Satuan": "Kategori (kode kelas)"},
])

display(deskripsi_kolom)
display(df.describe().T[["count", "mean", "std", "min", "50%", "max"]])

#### Keputusan saya
Saya memilih **Jalur A (Klasifikasi)** karena target `Output` merupakan variabel kategori diskret (`0`, `1`, dan `2`) dengan 880 baris dan 12 fitur prediktor, sehingga memenuhi syarat minimal dataset (>200 baris dan >=4 fitur) sekaligus relevan untuk dukungan keputusan pengelolaan kesuburan tanah.

--VISUALISASI DATA--

In [ ]:
# Grafik 1: Distribusi Target (Output) dan Distribusi Fitur Utama
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

target_counts = df["Output"].value_counts().sort_index()
sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title("Distribusi Kelas Target (Output)")
axes[0, 0].set_xlabel("Kelas Output")
axes[0, 0].set_ylabel("Jumlah Sampel")
for idx, val in enumerate(target_counts.values):
    axes[0, 0].text(idx, val + 8, f"{val} ({val/len(df)*100:.1f}%)", ha="center", fontsize=9)

fitur_plot = ["N", "P", "K", "pH", "EC"]
for ax, col in zip(axes.flat[1:], fitur_plot):
    sns.histplot(df[col], kde=True, ax=ax, bins=25, color="steelblue")
    ax.set_title(f"Distribusi {col}")

plt.tight_layout()
plt.savefig("../reports/01_distribusi_target_fitur.png")
plt.show()

In [ ]:
# Grafik 2: Hubungan Fitur dengan Target (Boxplot per Kelas) & Heatmap Korelasi
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df_melt = df[["N", "P", "K", "pH", "Output"]].melt(id_vars="Output", var_name="Fitur", value_name="Nilai")
sns.boxplot(data=df_melt, x="Fitur", y="Nilai", hue="Output", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Sebaran Fitur Utama per Kelas Output (Skala Log)")

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=axes[1])
axes[1].set_title("Matriks Korelasi Antar Variabel Numerik")

plt.tight_layout()
plt.savefig("../reports/02_korelasi_dan_boxplot.png")
plt.show()

print("Korelasi fitur terhadap Output:\n", corr["Output"].sort_values(ascending=False))

#### Keputusan saya
Saya menggunakan **StratifiedKFold** dan `stratify=y` pada pembagian data latih-uji karena kelas `Output` tidak seimbang (kelas `2` hanya 4,4%), serta mempertahankan seluruh 12 fitur numerik untuk tahap pra-pemrosesan dan pemodelan awal.

--PRA-PEMROSESSAN DATA--

In [ ]:
# Pemeriksaan missing value, duplikat, dan outlier (metode IQR)
print("Jumlah missing value per kolom:\n", df.isnull().sum())
print("Jumlah baris duplikat:", df.duplicated().sum())

fitur_cols = [c for c in df.columns if c != "Output"]
outlier_summary = []
for col in fitur_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_summary.append({
        "Fitur": col,
        "Q1": round(q1, 3),
        "Q3": round(q3, 3),
        "IQR": round(iqr, 3),
        "Batas Bawah": round(lower, 3),
        "Batas Atas": round(upper, 3),
        "Jumlah Outlier": n_out
    })

display(pd.DataFrame(outlier_summary))

# Split data latih (80%) dan data uji (20%) SEBELUM fit preprocessor (anti data leakage)
X = df[fitur_cols]
y = df["Output"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Ukuran X_train:", X_train.shape, "| Ukuran X_test:", X_test.shape)
print("Proporsi y_train:\n", y_train.value_counts(normalize=True).sort_index().round(4))
print("Proporsi y_test:\n", y_test.value_counts(normalize=True).sort_index().round(4))

# Pipeline pra-pemrosesan (di-fit hanya di dalam fold/data latih)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), fitur_cols)
    ]
)

#### Keputusan saya
- **Missing value & duplikat:** Dataset tidak memiliki *missing value* maupun baris duplikat, namun `SimpleImputer(strategy="median")` tetap dipasang di dalam `Pipeline` agar aman jika data baru mengandung nilai kosong.
- **Outlier (IQR):** *Outlier* terdeteksi pada beberapa fitur (misalnya `P`, `Zn`, `pH`, `OC`), tetapi tidak dihapus langsung karena bisa mencerminkan kondisi tanah spesifik di lapangan (terutama kelas minoritas `2`) dan berisiko mengurangi informasi kelas kecil.
- **Kategori & encoding:** Seluruh 12 fitur sudah numerik dan target `Output` sudah berupa kode kategori (`0`, `1`, `2`) tanpa inkonsistensi penulisan teks, sehingga tidak perlu *one-hot encoding* tambahan.
- **Split & scaling:** Saya membagi data 80:20 dengan `stratify=y` dan `random_state=SEED` (`515`) sebelum `fit`, lalu memakai `StandardScaler` di dalam `Pipeline` karena KNN dan SVM sensitif terhadap perbedaan skala satuan fitur.

--PEMODELAN--

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 1) Baseline: DummyClassifier
dummy_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))
])

# 2) Model utama 1: KNN (tuning nilai k dengan 5-fold Stratified CV)
knn_grid = GridSearchCV(
    estimator=Pipeline([
        ("prep", preprocessor),
        ("clf", KNeighborsClassifier())
    ]),
    param_grid={"clf__n_neighbors": [3, 5, 7, 9, 11, 15]},
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)
knn_grid.fit(X_train, y_train)

# 3) Model utama 2 (nilai tambah): SVM RBF (tuning C dan gamma dengan 5-fold Stratified CV)
svm_grid = GridSearchCV(
    estimator=Pipeline([
        ("prep", preprocessor),
        ("clf", SVC(kernel="rbf", random_state=SEED))
    ]),
    param_grid={
        "clf__C": [0.1, 1, 10, 50],
        "clf__gamma": ["scale", "auto"]
    },
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)
svm_grid.fit(X_train, y_train)

# 4) Model pembanding tambahan: Gaussian Naive Bayes
nb_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", GaussianNB())
])

print("Best KNN params:", knn_grid.best_params_, "| Best CV Macro-F1:", round(knn_grid.best_score_, 4))
print("Best SVM params:", svm_grid.best_params_, "| Best CV Macro-F1:", round(svm_grid.best_score_, 4))

# Tabel skor 5-fold CV pada data latih untuk seluruh model
models_cv = {
    "Dummy (most_frequent)": dummy_pipe,
    f"KNN (k={knn_grid.best_params_['clf__n_neighbors']})": knn_grid.best_estimator_,
    f"SVM RBF (C={svm_grid.best_params_['clf__C']}, gamma={svm_grid.best_params_['clf__gamma']})": svm_grid.best_estimator_,
    "GaussianNB": nb_pipe
}

cv_records = []
for nama, model in models_cv.items():
    acc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
    cv_records.append({
        "Model": nama,
        "CV Acc Fold 1-5": np.round(acc_scores, 4).tolist(),
        "Mean CV Acc": round(acc_scores.mean(), 4),
        "Std CV Acc": round(acc_scores.std(), 4),
        "Mean CV Macro-F1": round(f1_scores.mean(), 4),
        "Std CV Macro-F1": round(f1_scores.std(), 4),
    })

cv_df = pd.DataFrame(cv_records)
display(cv_df)

#### Keputusan saya
Saya menggunakan **DummyClassifier** (`most_frequent`) sebagai *baseline* wajib, **KNN** dan **SVM (RBF)** sebagai dua algoritma utama yang di-*tuning* dengan `GridSearchCV` (`scoring="f1_macro"`, 5-fold `StratifiedKFold`, `random_state=515`), serta **GaussianNB** sebagai pembanding tambahan. Pemilihan `f1_macro` pada *tuning* membantu menyeimbangkan performa pada kelas minoritas (`Output = 2`) di samping akurasi.

--EVALUASI DAN INTERPRETASI--

In [ ]:
# Evaluasi seluruh model satu kali pada data uji (X_test, y_test)
eval_rows = []
trained_models = {}

for nama, model in models_cv.items():
    model.fit(X_train, y_train)
    trained_models[nama] = model

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_f1_macro = f1_score(y_train, y_train_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test, y_test_pred, average="macro", zero_division=0)

    eval_rows.append({
        "Model": nama,
        "Train Acc": round(train_acc, 4),
        "Test Acc": round(test_acc, 4),
        "Selisih Acc (Train-Test)": round(train_acc - test_acc, 4),
        "Train Macro-F1": round(train_f1_macro, 4),
        "Test Macro-F1": round(test_f1_macro, 4),
        "Test Macro-Prec": round(precision_score(y_test, y_test_pred, average="macro", zero_division=0), 4),
        "Test Macro-Rec": round(recall_score(y_test, y_test_pred, average="macro", zero_division=0), 4),
    })

    print("=" * 65)
    print(f"MODEL: {nama}")
    print("Confusion Matrix (baris=aktual [0,1,2], kolom=prediksi [0,1,2]):")
    print(confusion_matrix(y_test, y_test_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))

eval_df = pd.DataFrame(eval_rows)
print("\n=== TABEL PERBANDINGAN MODEL VS BASELINE (DATA LATIH & UJI) ===")
display(eval_df)

# Visualisasi Mean CV Macro-F1 dan Confusion Matrix model terbaik (SVM RBF)
best_svm_name = [k for k in trained_models if k.startswith("SVM RBF")][0]
best_svm_pred = trained_models[best_svm_name].predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=cv_df, x="Mean CV Macro-F1", y="Model", ax=axes[0], palette="mako")
axes[0].set_title("Rata-rata Skor 5-Fold Stratified CV (Macro-F1)")
axes[0].set_xlim(0, 1)

ConfusionMatrixDisplay.from_predictions(
    y_test, best_svm_pred, display_labels=[0, 1, 2], cmap="Blues", ax=axes[1]
)
axes[1].set_title(f"Confusion Matrix Data Uji — {best_svm_name}")

plt.tight_layout()
plt.savefig("../reports/03_cv_dan_confusion_matrix.png")
plt.show()

 --KESIMPULAN--

### 6. Kesimpulan
- **Kelayakan model & pilihan akhir:** Model **SVM (RBF, `C=50, gamma='scale'`)** **layak dipakai** sebagai kandidat model klasifikasi dibanding *baseline* `DummyClassifier` (`most_frequent`) dengan dua dukungan angka utama:
  1. **Akurasi data uji** meningkat dari **0,5000** (50,00%) pada *baseline* menjadi **0,8693** (86,93%, yaitu 153 dari 176 sampel benar), serta **Mean CV Accuracy** meningkat dari **0,5000** menjadi **0,8282**.
  2. **Macro-F1 data uji** meningkat dari **0,2222** pada *baseline* menjadi **0,7673** (dan **Mean CV Macro-F1** dari **0,2222** menjadi **0,6251**), termasuk mampu mengenali kelas minoritas `2` (`F1 = 0,5333` pada data uji, sedangkan `Dummy` dan `KNN k=3` memperoleh `0,0000` pada kelas `2`).
- **Rekomendasi tindakan bagi pengguna:** Gunakan model **SVM RBF** sebagai alat bantu skrining awal klasifikasi tanah (`Output`), lalu verifikasi lanjutan di lapangan terutama pada sampel yang berada di dekat batas antar kelas atau diprediksi masuk kelas minoritas (`2`) sebelum menetapkan rekomendasi pemupukan/perlakuan lahan dalam skala besar.
- **Keterbatasan dan saran perbaikan:**
  - Ketimpangan kelas (`Output = 2` hanya 39 dari 880 baris) membuat metrik kelas `2` lebih sensitif terhadap sedikit kesalahan prediksi, dan nilai `C=50` menghasilkan *gap* antara skor latih dan uji.
  - Perbaikan selanjutnya dapat menguji nilai `C` yang lebih moderat (mis. `1`–`20`), menambahkan `class_weight="balanced"` pada `SVC`, serta menguji algoritma lain (seperti *Random Forest*) dengan validasi domain agronomi terhadap definisi label `0`, `1`, dan `2`.